### protein测试

In [1]:
import os
import re
import math
import glob
import argparse
import warnings
from collections import defaultdict
import pandas as pd

# ligand2中C24 C25 环O在vina输出的pose中的ATOM ID(需要在pymol中比对一下)
EPOXIDE_C24 = 29
EPOXIDE_O = 30
EPOXIDE_C25 = 31

# 核心催化原子:ASP101-OD1-OD2 150TYR-OH 230TYR-OH
CATALYTIC_ATOMS = {
    "D101_OD1": (101, "OD1"),
    "D101_OD2": (101, "OD2"),
    "Y150_OH":  (150, "OH"),
    "Y230_OH":  (230, "OH"),
}

# 用来过滤，未结合状态下的结合能
VINA_REF_STATE = 1.981

'''
功能:处理每个protein的pdbqt文件的字段
'''
def _parse_atom_line(ln):
    try:
        serial = int(ln[6:11].strip()) # 原子序列号
        name   = ln[12:16].strip() # 原子名字
        resn   = ln[17:20].strip() # 残基名字
        chain  = ln[21].strip() # 链
        resnum = int(ln[22:26].strip()) # 残基编号
        # 原子坐标
        x = float(ln[30:38].strip())
        y = float(ln[38:46].strip())
        z = float(ln[46:54].strip())
        return {
            "serial": serial, "name": name, "resn": resn,
            "chain": chain, "resnum": resnum,
            "x": x, "y": y, "z": z,
        }
    except (ValueError, IndexError):
        return None

'''
功能:找到对应的CATALYTIC_ATOMS中原子坐标
'''
def parse_receptor_catalytic(pdbqt_path):
    found = {}
    with open(pdbqt_path) as fh:
        for ln in fh:
            if not ln.startswith(("ATOM", "HETATM")):
                continue
            a = _parse_atom_line(ln)
            if a is None:
                continue
            # 
            key = (a["resnum"], a["name"])
            if key not in found:
                found[key] = (a["x"], a["y"], a["z"])
    coords = {}
    for col, (resnum, aname) in CATALYTIC_ATOMS.items():
        coords[col] = found.get((resnum, aname))
    return coords

In [2]:
pdbqtpath = "./10_MUTs_foldx_pdb2pqr_meeko/EPH_relaxed_0001_pdb2pqr_Repair_1_r2_0001.pdbqt"
result = parse_receptor_catalytic(pdbqtpath)

In [3]:
result

{'D101_OD1': (-1.08, 4.646, -1.312),
 'D101_OD2': (-2.657, 5.832, -0.331),
 'Y150_OH': (-4.356, 9.937, -3.137),
 'Y230_OH': (-1.381, 7.352, -4.973)}

### ligand测试

In [4]:
ligandpath = "./11_vina_results/EPH_relaxed_0001_pdb2pqr_Repair_1_r2_0001_out.pdbqt"

In [5]:
text = open(ligandpath).read()

In [6]:
print(text)

MODEL 1
REMARK VINA RESULT:      -7.5      0.000      0.000
REMARK VINA RESULT:     1.981      2.428      8.384
REMARK INTER + INTRA:           1.892
REMARK INTER:                   2.731
REMARK INTRA:                  -0.839
REMARK UNBOUND:                -0.611
REMARK SMILES C[C@H](CC[C@@H]1OC1(C)C)[C@H]1CC[C@@]2(C)[C@@H]3CC=C4[C@@H](CC[C@H](O)C4(C)C)[C@]3(C)CC[C@]12C
REMARK SMILES IDX 2 1 1 2 10 3 22 4 24 5 21 6 18 7 20 8 19 9 17 10 27 11 16 12
REMARK SMILES IDX 15 13 29 14 13 15 30 16 31 17 12 18 11 19 25 20 26 21 28 22
REMARK SMILES IDX 14 23 32 24 23 25 3 27 4 28 5 29 6 30 7 31 8 32 9 33
REMARK H PARENT 23 26
ROOT
ATOM      1  C   UNL     1      -2.019   9.157  -0.486  1.00  0.00    -0.011 C 
ATOM      2  C   UNL     1      -1.603   8.579  -1.841  1.00  0.00     0.008 C 
ENDROOT
BRANCH   1   3
ATOM      3  C   UNL     1      -1.149  10.361  -0.145  1.00  0.00    -0.002 C 
ATOM      4  C   UNL     1       1.241  17.693  -2.360  1.00  0.00     0.124 C 
ATOM      5  C   UNL     1   

In [ ]:
model_blocks = re.findall(r"^MODEL\s+(\d+)(.*?)^ENDMDL", text, re.S | re.M)
# model_blocks
for mnum_str, block in model_blocks:
    print(block)
    mnum = int(mnum_str)
    print(mnum)
    for rln in block.splitlines():
        if not rln.startswith("REMARK VINA RESULT"):
            continue
        parts = rln.split()
        print(parts)
        # parts: ['REMARK', 'VINA', 'RESULT:', aff, rmsd_lb, rmsd_ub]
        if len(parts) < 6:
            continue
        try:
            aff = float(parts[3])
            rlb = float(parts[4])
            rub = float(parts[5])
        except ValueError:
            continue
    # 取C24 C25 环氧的坐标
    atoms_by_serial = {}
    for ln in block.splitlines():
        if not ln.startswith(("ATOM", "HETATM")):
            continue
        a = _parse_atom_line(ln)
        if a is not None:
            atoms_by_serial[a["serial"]] = a
    print(atoms_by_serial)
    
    break


REMARK VINA RESULT:      -7.5      0.000      0.000
REMARK VINA RESULT:     1.981      2.428      8.384
REMARK INTER + INTRA:           1.892
REMARK INTER:                   2.731
REMARK INTRA:                  -0.839
REMARK UNBOUND:                -0.611
REMARK SMILES C[C@H](CC[C@@H]1OC1(C)C)[C@H]1CC[C@@]2(C)[C@@H]3CC=C4[C@@H](CC[C@H](O)C4(C)C)[C@]3(C)CC[C@]12C
REMARK SMILES IDX 2 1 1 2 10 3 22 4 24 5 21 6 18 7 20 8 19 9 17 10 27 11 16 12
REMARK SMILES IDX 15 13 29 14 13 15 30 16 31 17 12 18 11 19 25 20 26 21 28 22
REMARK SMILES IDX 14 23 32 24 23 25 3 27 4 28 5 29 6 30 7 31 8 32 9 33
REMARK H PARENT 23 26
ROOT
ATOM      1  C   UNL     1      -2.019   9.157  -0.486  1.00  0.00    -0.011 C 
ATOM      2  C   UNL     1      -1.603   8.579  -1.841  1.00  0.00     0.008 C 
ENDROOT
BRANCH   1   3
ATOM      3  C   UNL     1      -1.149  10.361  -0.145  1.00  0.00    -0.002 C 
ATOM      4  C   UNL     1       1.241  17.693  -2.360  1.00  0.00     0.124 C 
ATOM      5  C   UNL     1       1.3

In [21]:
receptor_dir = "10_MUTs_foldx_pdb2pqr_meeko"
out_dir = "11_vina_results"
rec_files = glob.glob(os.path.join(receptor_dir, "*.pdbqt"))
out_files = glob.glob(os.path.join(out_dir, "*_out.pdbqt"))
print(rec_files)
print(len(rec_files))

['10_MUTs_foldx_pdb2pqr_meeko\\EPH_relaxed_0001_pdb2pqr_Repair_100_r2_0001.pdbqt', '10_MUTs_foldx_pdb2pqr_meeko\\EPH_relaxed_0001_pdb2pqr_Repair_101_r2_0001.pdbqt', '10_MUTs_foldx_pdb2pqr_meeko\\EPH_relaxed_0001_pdb2pqr_Repair_102_r2_0001.pdbqt', '10_MUTs_foldx_pdb2pqr_meeko\\EPH_relaxed_0001_pdb2pqr_Repair_103_r2_0001.pdbqt', '10_MUTs_foldx_pdb2pqr_meeko\\EPH_relaxed_0001_pdb2pqr_Repair_104_r2_0001.pdbqt', '10_MUTs_foldx_pdb2pqr_meeko\\EPH_relaxed_0001_pdb2pqr_Repair_105_r2_0001.pdbqt', '10_MUTs_foldx_pdb2pqr_meeko\\EPH_relaxed_0001_pdb2pqr_Repair_106_r2_0001.pdbqt', '10_MUTs_foldx_pdb2pqr_meeko\\EPH_relaxed_0001_pdb2pqr_Repair_107_r2_0001.pdbqt', '10_MUTs_foldx_pdb2pqr_meeko\\EPH_relaxed_0001_pdb2pqr_Repair_108_r2_0001.pdbqt', '10_MUTs_foldx_pdb2pqr_meeko\\EPH_relaxed_0001_pdb2pqr_Repair_109_r2_0001.pdbqt', '10_MUTs_foldx_pdb2pqr_meeko\\EPH_relaxed_0001_pdb2pqr_Repair_10_r2_0001.pdbqt', '10_MUTs_foldx_pdb2pqr_meeko\\EPH_relaxed_0001_pdb2pqr_Repair_110_r2_0001.pdbqt', '10_MUTs_foldx_p

In [27]:
rec_map = {}  # basename -> path
for p in rec_files:
    base = os.path.splitext(os.path.basename(p))[0]
    print(base)
    rec_map[base] = p
    break

EPH_relaxed_0001_pdb2pqr_Repair_100_r2_0001


In [28]:
rec_map

{'EPH_relaxed_0001_pdb2pqr_Repair_100_r2_0001': '10_MUTs_foldx_pdb2pqr_meeko\\EPH_relaxed_0001_pdb2pqr_Repair_100_r2_0001.pdbqt'}

In [29]:
out_map = {}
for p in out_files:
    base = os.path.splitext(os.path.basename(p))[0]
    if base.endswith("_out"):
        base = base[:-4]
        print(base)
    out_map[base] = p
    break

EPH_relaxed_0001_pdb2pqr_Repair_100_r2_0001


In [30]:
out_map

{'EPH_relaxed_0001_pdb2pqr_Repair_100_r2_0001': '11_vina_results\\EPH_relaxed_0001_pdb2pqr_Repair_100_r2_0001_out.pdbqt'}